# 02 — Text Preprocessing & Duplicate-Aware Splitting

Notebook ini menghasilkan `train.csv`, `validation.csv`, `test.csv`, tokenizer bersama, dan konfigurasi sequence. Tokenizer hanya di-fit menggunakan teks training.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from tensorflow.keras.preprocessing.text import Tokenizer
from src.config import RAW_DATA_PATH, PROCESSED_DIR, RANDOM_SEED, CLASS_TO_ID, MAX_VOCAB_SIZE
from src.data_loader import load_dataset
from src.preprocessing import clean_text

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df = load_dataset(RAW_DATA_PATH)
print("Raw shape:", df.shape)
display(df.head())


## 1. Normalize text and validate labels

The initial preprocessing is intentionally conservative. It does not remove stopwords or apply stemming, so negation remains available to the models.

In [ ]:
before_rows = len(df)
df["review_clean"] = df["review"].apply(clean_text)
df["sentiment_norm"] = df["sentiment"].astype(str).str.strip().str.lower()
df = df[df["review_clean"].str.len() > 0].copy()
df = df[df["sentiment_norm"].isin(CLASS_TO_ID)].copy()
df["label_id"] = df["sentiment_norm"].map(CLASS_TO_ID).astype("int64")
print(f"Rows before filtering: {before_rows:,}")
print(f"Rows after removing empty/invalid text or labels: {len(df):,}")
print("Dropped rows:", before_rows - len(df))
print("Label mapping:", CLASS_TO_ID)
display(df["sentiment_norm"].value_counts().to_frame("count"))


## 2. Inspect repeated texts and label conflicts

Identical cleaned text is treated as one group during splitting. This avoids the same review wording appearing in multiple partitions. Conflicting labels are retained for inspection and will remain in the same split group.

In [ ]:
group_label_counts = df.groupby("review_clean")["sentiment_norm"].nunique()
conflicting_groups = group_label_counts[group_label_counts > 1]
print("Unique cleaned review groups:", df["review_clean"].nunique())
print("Groups with conflicting sentiment labels:", len(conflicting_groups))
if len(conflicting_groups):
    display(
        df[df["review_clean"].isin(conflicting_groups.index)]
        [["review", "sentiment", "app_name"] if "app_name" in df.columns else ["review", "sentiment"]]
        .head(20)
    )


## 3. Approximately 70/15/15 split using StratifiedGroupKFold

Twenty folds are assigned at review-text group level: 14 folds for train, 3 for validation, and 3 for test. Inspect the resulting proportions and class distributions; group sizes can make them differ slightly from the target.

In [ ]:
cv = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=RANDOM_SEED)
fold_id = np.full(len(df), -1, dtype=int)
X_dummy = np.zeros((len(df), 1))
y = df["label_id"].to_numpy()
groups = df["review_clean"].to_numpy()
for fold, (_, heldout_idx) in enumerate(cv.split(X_dummy, y, groups=groups)):
    fold_id[heldout_idx] = fold
if (fold_id < 0).any():
    raise RuntimeError("Some rows were not assigned to a split fold.")

df["split"] = np.select(
    [fold_id < 3, (fold_id >= 3) & (fold_id < 6)],
    ["test", "validation"],
    default="train",
)

# Verify that no normalized review text crosses partitions.
split_counts_per_group = df.groupby("review_clean")["split"].nunique()
assert int((split_counts_per_group > 1).sum()) == 0

summary = pd.crosstab(df["split"], df["sentiment_norm"], normalize="index")
print("Split proportions:")
display(df["split"].value_counts(normalize=True).rename("proportion").to_frame())
print("Class proportions by split:")
display(summary)


## 4. Save split manifests

Keep source columns for auditability. The model notebooks should use `review_clean` and `label_id` only.

In [ ]:
for split_name in ["train", "validation", "test"]:
    part = df[df["split"] == split_name].copy()
    output_path = PROCESSED_DIR / f"{split_name}.csv"
    part.to_csv(output_path, index=False)
    print(f"Saved {len(part):,} rows -> {output_path}")


## 5. Fit tokenizer on training text only

The tokenizer and maximum sequence length are shared by LSTM and GRU. Validation and test are never used in `fit_on_texts`.

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["review_clean"].astype(str).tolist())

train_sequences = tokenizer.texts_to_sequences(train_df["review_clean"].astype(str).tolist())
lengths = np.array([len(seq) for seq in train_sequences if len(seq) > 0])
max_length = max(5, int(np.ceil(np.percentile(lengths, 95))))

tokenizer_path = PROCESSED_DIR / "tokenizer.json"
tokenizer_path.write_text(tokenizer.to_json(), encoding="utf-8")
sequence_config = {
    "max_vocab_size": MAX_VOCAB_SIZE,
    "vocab_size_observed": min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1),
    "max_length": max_length,
    "padding": "post",
    "truncating": "post",
    "tokenizer_fit_partition": "train",
    "random_seed": RANDOM_SEED,
}
(PROCESSED_DIR / "sequence_config.json").write_text(
    json.dumps(sequence_config, indent=2), encoding="utf-8"
)
print("Sequence config:", sequence_config)


## 6. Save final split statistics

Keep this output with the report. Review it before training. If a class is missing from a split or a large discrepancy appears, investigate the grouping and split rather than ignoring it.

In [ ]:
split_stats = (
    df.groupby(["split", "sentiment_norm"]).size()
    .rename("count").reset_index()
)
split_stats.to_csv(PROCESSED_DIR / "split_label_counts.csv", index=False)
display(split_stats)
print("Preprocessing artifacts saved in:", PROCESSED_DIR)
